# Fra API-søk til tabell, hent forhåndsvalgt uttrekk for mange tabeller

### Introduksjon
1. I PxWebApi v2 er det mulig å hente et default datasett med en URL uten spørreparametre. Default språk Norsk og outputformat JSON-stat2 er satt i [/config](https://data.ssb.no/api/pxwebapi/v2/config)
2. i beta versjonen til v2 ville /navigation gi mulighet til å angi kortnavn for en statstikk direkte uten å traversere treet. Men /navigation i v2 er helt tatt vekk. Nå  må en bruke søk istedet.
3. Ny parameter i søk er discontinued, Ved å bruke `discontinued=false` ekskluderer en avslutta tabeller, dvs. de som ikke oppdateres. Det er over halvparten av statistikkbanktabellene. 

Eksempelet henter JSON-stat uten det dedikerte biblioteket pyjstat. Her får en variabelkodene, de tilhørende tekstene vises som _label.

Takk til [Carl Corneill](https://github.com/aecorn) for eksempel på funksjoner, som jeg har bearbeidet litt bl.a. ved hjelp av Claude.

In [1]:
import requests
import pandas as pd
import itertools # itertools is a part of Python
import time # for å være hensynsfull

Søk etter et ord i tittel på en Statistikkbanktabell

In [2]:
query = 'title:Konsumprisindeks'

In [3]:
url = f"https://data.ssb.no/api/pxwebapi/v2/tables?query={query}&includeDiscontinued=false&pagesize=30"

Vi henter inntil 30 treff, ved å øke `pagesize=30`.

In [4]:
response = requests.get(url)

In [5]:
data = response.json()

Fjern # for å se hele søkeresultatet

In [6]:
# data['tables']

Vis titler og sist oppdatert

In [7]:
for table in data['tables']:
    print(f"{table['label']} : {table['updated']}")

14700: Konsumprisindeks, etter vare- og tjenestegruppe (2025=100) 2000M01-2026M01 : 2026-02-10T07:00:00Z
14708: Sesongjustert konsumprisindeks og KPI-JAE (2025=100) 1985M01-2026M01 : 2026-02-10T07:00:00Z
14709: Konsumprisindeks, historisk serie, etter måned (2025=100) 1920-2026 : 2026-02-10T07:00:00Z
14710: Konsumprisindeks, historisk serie (2025=100) 1920M03-2026M01 : 2026-02-10T07:00:00Z
14701: Konsumprisindeks, etter vare- og tjenestegruppe (2025=100) 2000-2025 : 2026-02-10T07:00:00Z
14711: Konsumprisindeks, historisk serie (2025=100) 1865-2025 : 2026-02-06T07:00:00Z
14696: Harmonisert konsumprisindeks for Norge, etter vare- og tjenestegruppe (2025=100) 2000M01-2026M01 : 2026-02-10T07:00:00Z
14697: Harmonisert konsumprisindeks for Norge, etter vare- og tjenestegruppe (2025=100) 2000-2025 : 2026-02-09T07:00:00Z
14702: Konsumprisindeks (KPI, KPI-JA og KPI-JAE), etter leveringssektor (2025=100) 2015M01-2026M01 : 2026-02-10T07:00:00Z
14704: Justert konsumprisindeks (KPI-JA og KPI-JAE), 

In [8]:
# Hent ut alle table IDs
table_ids = [table['id'] for table in data['tables']]

In [9]:
table_ids

['14700',
 '14708',
 '14709',
 '14710',
 '14701',
 '14711',
 '14696',
 '14697',
 '14702',
 '14704',
 '14703',
 '14705',
 '14698',
 '14699',
 '14706',
 '14707']

Funksjon for å hente en tabell fra APIv2. 

JSON-stat bruker metoden [Row-major-order](https://en.wikipedia.org/wiki/Row-_and_column-major_order) for lagring av data

In [10]:
def fetch_data_and_metadata(tableid: str) -> dict:
    response = requests.get(f'https://data.ssb.no/api/pxwebapi/v2/tables/{tableid}/data')

    # the whole dataset as JSON-stat2
    js2_data = response.json()

    # Get the dimensions of the dataset
    dimensions = {
            dim_name: list(dim_info["category"]["index"].keys()) 
            for dim_name, dim_info in js2_data["dimension"].items()
        }
    # Get the code to the dataframe
    dimension_combinations = list(itertools.product(*list(dimensions.values())))
    df = pd.DataFrame(dimension_combinations, columns=dimensions.keys())
    
    # Get the textlabels and add them next to codecolumn
    dimensions_labels = {
            dim_name: dim_info["category"]["label"] 
            for dim_name, dim_info in js2_data["dimension"].items()
        }
    for i, (col, codelist) in enumerate(dimensions_labels.items()):
            label_colname = f"{col}_label"
            df.insert((i*2+1), label_colname, df[col].map(codelist))
    
    # Add values to the dataframe
    df["Value"] = js2_data["value"]

    # Get metadata from dataset 
    metadata = dict(
        table_id = js2_data['extension']['px']['tableid'],
        short_title = js2_data['extension']['px']['contents'],
        title = js2_data['label'],
        source = js2_data['source'],
        last_update = js2_data['updated'],
        # include footnote, needs post processing
        # note = js2_data['note']
        )
        
    return {
        'dataframe': df, 
        'metadata' : metadata
        }

Henter default datasett for  KPI-tabellen, med ID 14706

In [11]:
fetch_data_and_metadata('14706')['dataframe']


,KPIavledetSerie,KPIavledetSerie_label,Tid,Tid_label,ContentsCode,ContentsCode_label,Value
0,KPI,Konsumprisindeksen totalt (KPI),2026M01,2026M01,KPIJustIndMnd,Indeks (2025=100),101.6
1,KPI,Konsumprisindeksen totalt (KPI),2026M01,2026M01,Manedsendring,Månedsendring (prosent),0.6
2,KPI,Konsumprisindeksen totalt (KPI),2026M01,2026M01,Tolvmanedersendring,12-måneders endring (prosent),3.6
3,KPI-JE,KPI uten energivarer (KPI-JE),2026M01,2026M01,KPIJustIndMnd,Indeks (2025=100),101.4
4,KPI-JE,KPI uten energivarer (KPI-JE),2026M01,2026M01,Manedsendring,Månedsendring (prosent),0.5
5,KPI-JE,KPI uten energivarer (KPI-JE),2026M01,2026M01,Tolvmanedersendring,12-måneders endring (prosent),3.6
6,KPI-JEL,KPI uten elektrisitet (KPI-JEL),2026M01,2026M01,KPIJustIndMnd,Indeks (2025=100),101.3
7,KPI-JEL,KPI uten elektrisitet (KPI-JEL),2026M01,2026M01,Manedsendring,Månedsendring (prosent),0.4
8,KPI-JEL,KPI uten elektrisitet (KPI-JEL),2026M01,2026M01,Tolvmanedersendring,12-måneders endring (prosent),3.4
9,KPI-JA,KPI justert for avgiftsendringer (KPI-JA),2026M01,2026M01,KPIJustIndMnd,Indeks (2025=100),102.8


Henter JSON-stat2 metadata for tabellen

In [12]:
fetch_data_and_metadata('14706')['metadata']

{'table_id': '14706',
 'short_title': '14706: Konsumprisindeks,',
 'title': '14706: Konsumprisindeks, etter avledet serie, måned og statistikkvariabel',
 'source': 'Statistisk sentralbyrå',
 'last_update': '2026-02-10T07:00:00Z'}

### Hent alle tabeller med metadata og vis 2 første linjer av datasett.

In [13]:
for tableid in table_ids:
    result = fetch_data_and_metadata(tableid)
    print(result['metadata'])
    display(result['dataframe'].head(2))
    time.sleep(2) # ønsker å ikke belaste SSB API-et for mye
    

{'table_id': '14700', 'short_title': '14700: Konsumprisindeks,', 'title': '14700: Konsumprisindeks, etter vare- og tjenestegruppe, måned og statistikkvariabel', 'source': 'Statistisk sentralbyrå', 'last_update': '2026-02-10T07:00:00Z'}


,VareTjenesteGrp,VareTjenesteGrp_label,Tid,Tid_label,ContentsCode,ContentsCode_label,Value
0,00,I alt,2026M01,2026M01,KpiIndMnd,Konsumprisindeks (2025=100),101.6
1,00,I alt,2026M01,2026M01,Manedsendring,Månedsendring (prosent),0.6


{'table_id': '14708', 'short_title': '14708: Sesongjustert  konsumprisindeks (2025=100),', 'title': '14708: Sesongjustert  konsumprisindeks (2025=100), etter avledet serie, statistikkvariabel og måned', 'source': 'Statistisk sentralbyrå', 'last_update': '2026-02-10T07:00:00Z'}


,KPIavledetSerie,KPIavledetSerie_label,ContentsCode,ContentsCode_label,Tid,Tid_label,Value
0,KPI,Konsumprisindeksen totalt (KPI),KPIsesong,Sesongjustert indeks,2025M01,2025M01,98.5
1,KPI,Konsumprisindeksen totalt (KPI),KPIsesong,Sesongjustert indeks,2025M02,2025M02,99.8


{'table_id': '14709', 'short_title': '14709: Konsumprisindeks (2025=100),', 'title': '14709: Konsumprisindeks (2025=100), etter måned, statistikkvariabel og år', 'source': 'Statistisk sentralbyrå', 'last_update': '2026-02-10T07:00:00Z'}


...

,VareTjenesteGrp,VareTjenesteGrp_label,ContentsCode,ContentsCode_label,Tid,Tid_label,Value
0,00,I alt,HkpilndMnd,Harmonisert konsumprisindeks med konstante avg...,2013,2013,70.2
1,00,I alt,HkpilndMnd,Harmonisert konsumprisindeks med konstante avg...,2014,2014,71.4


{'table_id': '14706', 'short_title': '14706: Konsumprisindeks,', 'title': '14706: Konsumprisindeks, etter avledet serie, måned og statistikkvariabel', 'source': 'Statistisk sentralbyrå', 'last_update': '2026-02-10T07:00:00Z'}


,KPIavledetSerie,KPIavledetSerie_label,Tid,Tid_label,ContentsCode,ContentsCode_label,Value
0,KPI,Konsumprisindeksen totalt (KPI),2026M01,2026M01,KPIJustIndMnd,Indeks (2025=100),101.6
1,KPI,Konsumprisindeksen totalt (KPI),2026M01,2026M01,Manedsendring,Månedsendring (prosent),0.6


{'table_id': '14707', 'short_title': '14707: Konsumprisindeks,', 'title': '14707: Konsumprisindeks, etter avledet serie, år og statistikkvariabel', 'source': 'Statistisk sentralbyrå', 'last_update': '2026-02-08T07:00:00Z'}


,KPIavledetSerie,KPIavledetSerie_label,Tid,Tid_label,ContentsCode,ContentsCode_label,Value
0,KPI,Konsumprisindeksen totalt (KPI),2025,2025,KPIJaar,Indeks (2025=100),100.0
1,KPI,Konsumprisindeksen totalt (KPI),2025,2025,Aarsendring,Årsendring (prosent),3.0


In [14]:
summary = pd.DataFrame([
    fetch_data_and_metadata(tid)['metadata']  # allerede hentet, burde caches
    for tid in table_ids
])

In [15]:
# øker max kolonnebredde
pd.options.display.max_colwidth=85

In [16]:
summary

,table_id,short_title,title,source,last_update
0,14700,"14700: Konsumprisindeks,","14700: Konsumprisindeks, etter vare- og tjenestegruppe, måned og statistikkvariabel",Statistisk sentralbyrå,2026-02-10T07:00:00Z
1,14708,"14708: Sesongjustert konsumprisindeks (2025=100),","14708: Sesongjustert konsumprisindeks (2025=100), etter avledet serie, statistik...",Statistisk sentralbyrå,2026-02-10T07:00:00Z
2,14709,"14709: Konsumprisindeks (2025=100),","14709: Konsumprisindeks (2025=100), etter måned, statistikkvariabel og år",Statistisk sentralbyrå,2026-02-10T07:00:00Z
3,14710,"14710: Konsumprisindeks (2025=100),","14710: Konsumprisindeks (2025=100), etter måned og statistikkvariabel",Statistisk sentralbyrå,2026-02-10T07:00:00Z
4,14701,"14701: Konsumprisindeks,","14701: Konsumprisindeks, etter vare- og tjenestegruppe, år og statistikkvariabel",Statistisk sentralbyrå,2026-02-10T07:00:00Z
5,14711,"14711: Konsumprisindeks (2025=100),","14711: Konsumprisindeks (2025=100), etter år og statistikkvariabel",Statistisk sentralbyrå,2026-02-06T07:00:00Z
6,14696,"14696: Harmonisert konsumprisindeks,","14696: Harmonisert konsumprisindeks, etter vare- og tjenestegruppe, måned og stat...",Statistisk sentralbyrå,2026-02-10T07:00:00Z
7,14697,"14697: Harmonisert konsumprisindeks,","14697: Harmonisert konsumprisindeks, etter vare- og tjenestegruppe, år og statist...",Statistisk sentralbyrå,2026-02-09T07:00:00Z
8,14702,"14702: Konsumprisindeks,","14702: Konsumprisindeks, etter leveringssektor, statistikkvariabel, måned og avle...",Statistisk sentralbyrå,2026-02-10T07:00:00Z
9,14704,"14704: Konsumprisindeks,","14704: Konsumprisindeks, etter vare- og tjenestegruppe, statistikkvariabel, måned...",Statistisk sentralbyrå,2026-02-10T07:00:00Z
